# Used Cars Data Preparation
**Price Prediction — Data Cleaning, Encoding & Baseline MAE**

> **Dataset:** CarDekho Used Car Data (manishkr1754/cardekho-used-car-data on Kaggle)  
> The notebook tries the Kaggle source first, then falls back to a local/sample CSV.


## Task 1 — Load, Inspect, and Clean

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# ── Load dataset ──────────────────────────────────────────────────────────────
# Option 1: Try the Kaggle dataset via kagglehub (requires kaggle credentials)
try:
    import kagglehub, os, glob
    path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")
    print("Path to dataset files:", path)
    csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
    print("CSV files found:", csv_files)
    df = pd.read_csv(csv_files[0])
    print(f"✅ Loaded from Kaggle: {csv_files[0]}")
except Exception as e:
    print(f"⚠️  Kaggle download failed ({type(e).__name__}): {e}")
    print("Falling back to GitHub URL ...")
    try:
        url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/used_cars.csv"
        df = pd.read_csv(url)
        print("✅ Loaded from GitHub URL")
    except Exception as e2:
        print(f"⚠️  GitHub URL also failed: {e2}")
        # Final fallback — place your CSV in the same folder as this notebook
        df = pd.read_csv("used_cars.csv")
        print("✅ Loaded from local file: used_cars.csv")


Path to dataset files: C:\Users\vemur\.cache\kagglehub\datasets\manishkr1754\cardekho-used-car-data\versions\2
CSV files found: ['C:\\Users\\vemur\\.cache\\kagglehub\\datasets\\manishkr1754\\cardekho-used-car-data\\versions\\2\\cardekho_dataset.csv']
✅ Loaded from Kaggle: C:\Users\vemur\.cache\kagglehub\datasets\manishkr1754\cardekho-used-car-data\versions\2\cardekho_dataset.csv


In [6]:
# ── 1. Initial inspection ─────────────────────────────────────────────────────
print("Shape:", df.shape)
print()
df.info()


Shape: (15411, 14)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB


In [7]:
print("\nMissing value % per column:")
missing_pct = df.isnull().sum() * 100 / len(df)
print(missing_pct.sort_values(ascending=False).to_string())



Missing value % per column:
Unnamed: 0           0.0
car_name             0.0
brand                0.0
model                0.0
vehicle_age          0.0
km_driven            0.0
seller_type          0.0
fuel_type            0.0
transmission_type    0.0
mileage              0.0
engine               0.0
max_power            0.0
seats                0.0
selling_price        0.0


In [8]:
# ── 2. Drop rows where selling_price is missing ───────────────────────────────
before = len(df)
df = df.dropna(subset=['selling_price'])
print(f"Dropped {before - len(df)} rows with missing selling_price → {df.shape}")


Dropped 0 rows with missing selling_price → (15411, 14)


In [9]:
# ── 3. Strip unit strings, convert to numeric, fill NaN with median ───────────
def strip_units_and_convert(series):
    """
    Extract the leading numeric part from strings such as:
      '23.1 kmpl'  →  23.1
      '1197 CC'    →  1197.0
      '88.50 bhp'  →  88.5
    Non-parseable values become NaN (coerced).
    """
    return pd.to_numeric(
        series.astype(str).str.extract(r'([\d.]+)')[0],
        errors='coerce'
    )

for col in ['mileage', 'engine', 'max_power']:
    df[col] = strip_units_and_convert(df[col])
    col_median = df[col].median()
    df[col] = df[col].fillna(col_median)
    print(f"  {col:12s}  median = {col_median:>10.2f}  |  nulls after fill = {df[col].isnull().sum()}")


  mileage       median =      19.67  |  nulls after fill = 0
  engine        median =    1248.00  |  nulls after fill = 0
  max_power     median =      88.50  |  nulls after fill = 0


In [10]:
# ── 4. Remove placeholder / extreme prices ────────────────────────────────────
before = len(df)
df = df[df['selling_price'] != 999_999_999]
df = df[df['selling_price'] >= 10_000]
print(f"Removed {before - len(df)} rows with outlier selling_price → {len(df)} rows remain")


Removed 0 rows with outlier selling_price → 15411 rows remain


In [11]:
# ── 5. Drop exact duplicate rows ─────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows.")

print("\n✅ Final shape after all cleaning:")
print(df.shape)


Dropped 0 duplicate rows.

✅ Final shape after all cleaning:
(15411, 14)


## Task 2 — Encode Categorical Features

In [12]:
# ── 1. Label encode transmission_type ────────────────────────────────────────
df['transmission_type'] = df['transmission_type'].map({'Manual': 0, 'Automatic': 1})
print("transmission_type value counts:")
print(df['transmission_type'].value_counts())


transmission_type value counts:
transmission_type
0    12225
1     3186
Name: count, dtype: int64


In [13]:
# ── 2. One-hot encode fuel_type and seller_type ───────────────────────────────
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

# Newer pandas returns bool dtype for dummy columns — cast to int for consistency
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("Shape after one-hot encoding:", df.shape)


Shape after one-hot encoding: (15411, 18)


In [14]:
# ── 3. Final column list ──────────────────────────────────────────────────────
print("Columns in final DataFrame:")
print(df.columns.tolist())


Columns in final DataFrame:
['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


### Why `drop_first=True`?

**Avoiding the dummy variable trap (multicollinearity)**

When one-hot encoding a categorical column with *k* categories, only **k − 1** binary columns are needed to uniquely represent all values. The removed category is implicitly captured when all remaining dummies equal `0`.

**Example — `fuel_type` with categories `[CNG, Diesel, Electric, LPG, Petrol]` (`CNG` dropped):**

| fuel_type_Diesel | fuel_type_Electric | fuel_type_LPG | fuel_type_Petrol | Encoded as |
|:---:|:---:|:---:|:---:|:---|
| 1 | 0 | 0 | 0 | Diesel |
| 0 | 0 | 0 | 1 | Petrol |
| **0** | **0** | **0** | **0** | **CNG** ← reference / dropped category |

**What does a row of all zeros mean?**  
A row where every dummy column for a group is `0` represents the **reference (baseline) category** — whichever category was dropped by `drop_first=True` (the first one alphabetically). The model learns the effect of every other category *relative to this baseline*.

Keeping all *k* columns creates **perfect multicollinearity**: one column is always a linear combination of the others, causing a singular matrix in linear models, undefined coefficients, and inflated standard errors in regularised models.


## Task 3 — Split and Compute Baseline MAE

In [18]:
# ── 1. Define X and y; drop non-numeric columns ───────────────────────────────
y = df['selling_price']
X = df.drop(columns=['selling_price'])

non_numeric = X.select_dtypes(exclude='number').columns.tolist()
if non_numeric:
    print(f"Dropping non-numeric columns: {non_numeric}")
    X = X.drop(columns=non_numeric)
else:
    print("No non-numeric columns to drop.")

print(f"\nX shape : {X.shape}")
print(f"y shape : {y.shape}")


Dropping non-numeric columns: ['car_name', 'brand', 'model']

X shape : (15411, 14)
y shape : (15411,)


In [16]:
# ── 2. 80 / 20 train-test split ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train size : {X_train.shape[0]} rows")
print(f"Test  size : {X_test.shape[0]} rows")


Train size : 12328 rows
Test  size : 3083 rows


In [17]:
# ── 3 & 4. Baseline MAE — always predict the training-set mean ───────────────
baseline_pred = np.full(shape=len(y_test), fill_value=y_train.mean())
baseline_mae  = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: ₹{round(baseline_mae):,}")


Baseline MAE: ₹468,748


### Interpretation
The **Baseline MAE** is the error produced by the simplest possible model: always predict the average training price.  
Any real model — linear regression, random forest, gradient boosting — trained on these features should achieve a **lower MAE** than this baseline. It serves as the minimum performance bar the assignment requires us to beat in future tasks.
